# Football Tracking Pipeline — Offline Global Re-Linking

Single pipeline, one pass per stage, no online/bounded-delay correction:

1. Run detection + BoT-SORT once, causally -- gives short, locally-reliable **tracklets** (a
   tracklet = one continuous stretch where a raw track ID stayed alive with no ambiguity).
2. **Deliberately split** a tracklet at every point where it was in contact (same-class bbox
   overlap) with another track -- tracker continuity is trusted *between* occlusions, never
   *across* one, since that's exactly where an in-place identity swap can happen invisibly.
3. Solve **one global assignment problem per class, over the whole match at once** (Hungarian
   algorithm): which tracklet is immediately followed by which, using appearance similarity +
   motion consistency, with a hard "can't link if they were ever alive at the same time"
   constraint. This is what a bounded-delay online corrector structurally cannot do -- it can
   only ever look at a small sliding window, while this uses the entire match to resolve every
   link.
4. Union the accepted links into final chains -> final global identities, apply ratio-aware
   class locking, save, annotate.

Run top to bottom. Fill in paths in Cell 1.


## Cell 1 — Config

In [22]:
import os
import cv2
import pickle
import colorsys
import numpy as np
from pathlib import Path
from collections import defaultdict, deque
from scipy.optimize import linear_sum_assignment

DETECTION_CACHE_PATH = "/kaggle/input/datasets/zeinsaad/osnet-sportsmot/barca_atletico_detection_cache_v2.pkl"
OSNET_WEIGHTS_PATH    = "/kaggle/input/datasets/zeinsaad/osnet-sportsmot/osnet_x1_0_sportsmot_best.pt"
VIDEO_PATH             = "/kaggle/input/datasets/zeinsaad/osnet-sportsmot/clip.mkv"

OUTPUT_TRACKING_CACHE_PATH = "barca_atletico_tracking_cache_final.pkl"
OUTPUT_VIDEO_PATH = "barca_atletico_annotated_final.mp4"

DEVICE = 0   # Kaggle GPU index, or "cpu"

CLASS_TO_ID = {"player": 0, "goalkeeper": 1, "referee": 2}
ID_TO_CLASS = {v: k for k, v in CLASS_TO_ID.items()}

# ---- tracklet splitting: force a split at every same-class contact, so no identity is ever
# trusted to survive an occlusion just because the raw tracker ID didn't change ----
CONTACT_IOU_THRESH = 0.3

# ---- pre-link noise filtering: drop tracklets shorter than this BEFORE linking (too short to
# carry reliable appearance/motion signal). Much smaller than the final ghost-track filter below,
# since a legitimate player might be genuinely split into several short chunks by contact events,
# each of which should still be linkable. ----
MIN_TRACKLET_LEN = 20

# ---- global linking parameters ----
MAX_LINK_GAP = 500       # max frames between one tracklet ending and the next starting to be
                          # considered linkable at all. Raised from an initial 300 after finding
                          # a side referee's fragments split by more than 300 frames (long
                          # stretches near the touchline, out of frame). If you still see extra
                          # split identities for a class with a known small headcount, raise this
                          # further and/or lower MIN_LINK_SCORE -- check the [link] debug output
                          # to see whether candidate pairs are rejected on score or never
                          # considered at all (gap exceeded MAX_LINK_GAP).
MIN_LINK_SCORE = 0.3     # minimum combined appearance+motion score to accept a link
EMBED_WINDOW = 8         # frames used for head/tail embedding averaging
MOTION_WEIGHT = 0.3      # weight of the motion penalty in the combined score
MOTION_NORM_PX = 300.0   # pixel distance that saturates the motion penalty to 1.0

# ---- final ghost-track filter + ratio-aware class locking ----
MIN_TRACK_LENGTH = 300
MIN_CONFIRM_FRAMES_ABS = 10
MIN_CONFIRM_RATIO = 0.02
MAX_IDS_PER_CLASS_EXPECTED = {"goalkeeper": 2, "referee": 3}   # sanity check only, not enforced

DEBUG_LINKING = True

print("Checking paths...")
for p in [DETECTION_CACHE_PATH, OSNET_WEIGHTS_PATH, VIDEO_PATH]:
    print(p, "->", "OK" if os.path.exists(p) else "MISSING")


Checking paths...
/kaggle/input/datasets/zeinsaad/osnet-sportsmot/barca_atletico_detection_cache_v2.pkl -> OK
/kaggle/input/datasets/zeinsaad/osnet-sportsmot/osnet_x1_0_sportsmot_best.pt -> OK
/kaggle/input/datasets/zeinsaad/osnet-sportsmot/clip.mkv -> OK


## Cell 2 — Load fine-tuned OSNet + detection cache

In [23]:
!pip install -q torchreid

In [24]:
import torch, torch.nn as nn
import torchreid
import torchvision.transforms as T
from PIL import Image

ckpt = torch.load(OSNET_WEIGHTS_PATH, map_location="cpu")
state_dict = ckpt.get("state_dict", ckpt.get("model", ckpt)) if isinstance(ckpt, dict) else ckpt

osnet = torchreid.models.build_model(name="osnet_x1_0", num_classes=302, pretrained=False)
osnet.load_state_dict(state_dict, strict=True)
osnet.classifier = nn.Identity()
osnet.eval()
osnet.to("cuda" if DEVICE != "cpu" else "cpu")

osnet_transform = T.Compose([
    T.Resize((256, 128)), T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

with open(DETECTION_CACHE_PATH, "rb") as f:
    detection_cache = pickle.load(f)

print("OSNet ready. Embedding dim: 512")
print(f"Detection cache loaded: {len(detection_cache)} frames")


OSNet ready. Embedding dim: 512
Detection cache loaded: 3001 frames


## Cell 3 — Install boxmot (pinned, `--no-deps`) and initialize tracker
**Never re-run `pip install boxmot` without `--no-deps`** -- it will downgrade torch/numpy.

In [25]:
!pip install -q boxmot==10.0.84 --no-deps
!pip install -q loguru ftfy regex lap filterpy --no-deps


In [26]:
from boxmot import BoTSORT

device_str = "cpu" if DEVICE == "cpu" else str(DEVICE)

tracker = BoTSORT(
    model_weights=Path(OSNET_WEIGHTS_PATH),
    device=device_str,
    fp16=False,
    track_high_thresh=0.5,
    track_low_thresh=0.1,
    new_track_thresh=0.6,
    track_buffer=100,
    match_thresh=0.8,
    proximity_thresh=0.5,
    appearance_thresh=0.25,
    cmc_method="sof",
    frame_rate=25,
)
print(f"BoT-SORT initialized on device='{device_str}'.")


2026-07-04 19:38:32.379 | INFO     | boxmot.utils.torch_utils:select_device:52 - Yolo Tracking v10.0.83 🚀 Python-3.12.13 torch-2.10.0+cu128
CUDA:0 (Tesla T4, 14912MiB)
2026-07-04 19:38:33.093 | SUCCESS  | boxmot.appearance.reid_model_factory:load_pretrained_weights:183 - Loaded pretrained weights from /kaggle/input/datasets/zeinsaad/osnet-sportsmot/osnet_x1_0_sportsmot_best.pt


BoT-SORT initialized on device='0'.


## Cell 4 — Core functions: tracklet building, contact splitting, feature computation, global linking
Defined once, upfront, and validated against synthetic data in Cell 5 before ever touching the
real video.

In [27]:
def iou(box1, box2):
    xa1, ya1, xa2, ya2 = box1; xb1, yb1, xb2, yb2 = box2
    ix1, iy1 = max(xa1, xb1), max(ya1, yb1)
    ix2, iy2 = min(xa2, xb2), min(ya2, yb2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    union = (xa2 - xa1) * (ya2 - ya1) + (xb2 - xb1) * (yb2 - yb1) - inter
    return inter / union if union > 0 else 0.0


def bbox_center(bbox):
    x1, y1, x2, y2 = bbox
    return np.array([(x1 + x2) / 2, (y1 + y2) / 2])


def make_tracklet(entries):
    """entries: list of (frame_idx, bbox, conf, cls, embedding), sorted by frame_idx."""
    frames = [e[0] for e in entries]
    bbox_by_frame = {e[0]: e[1] for e in entries}
    conf_by_frame = {e[0]: e[2] for e in entries}
    class_by_frame = {e[0]: e[3] for e in entries}
    embedding_by_frame = {e[0]: e[4] for e in entries}
    classes = [e[3] for e in entries]
    majority_class = max(set(classes), key=classes.count)
    return {
        "frames": frames, "bbox_by_frame": bbox_by_frame, "conf_by_frame": conf_by_frame,
        "class_by_frame": class_by_frame, "embedding_by_frame": embedding_by_frame,
        "class": majority_class,
    }


def build_initial_tracklets(raw_tracks_by_frame):
    """Groups raw per-frame tracker output by raw_track_id. A raw ID's lifespan is already
    'one continuous stretch with no ambiguity according to the tracker' -- exactly what a
    tracklet should be before contact-based splitting."""
    raw_entries_by_id = defaultdict(list)
    for frame_idx, dets in raw_tracks_by_frame.items():
        for d in dets:
            raw_entries_by_id[d["raw_track_id"]].append(
                (frame_idx, d["bbox"], d["conf"], d["class"], d["embedding"])
            )

    tracklets = []
    for raw_id, entries in raw_entries_by_id.items():
        entries.sort(key=lambda e: e[0])
        segments, current = [], [entries[0]]
        for e in entries[1:]:
            if e[0] == current[-1][0] + 1:
                current.append(e)
            else:
                segments.append(current)
                current = [e]
        segments.append(current)
        for seg in segments:
            tracklets.append(make_tracklet(seg))
    return tracklets


def find_contact_split_points(tracklets, iou_thresh, merge_gap=1):
    """Returns {tracklet_index: set of cut-boundary frames}. A cut boundary at frame f means
    'end a chunk after frame f' -- so the tracklet is split right before and right after each
    contact event, isolating the ambiguous contact frames into their own short segment instead
    of leaving them attached to a clean pre/post segment.

    Contact frames are grouped into contiguous runs (allowing gaps of up to merge_gap frames to
    still count as one run) before computing boundaries -- otherwise an extended contact (many
    consecutive overlapping frames) would get a cut point at EVERY overlapping frame, shredding
    the tracklet into single-frame fragments that then get dropped entirely by MIN_TRACKLET_LEN,
    silently deleting real player data for that whole stretch.
    """
    overlap_frames_by_pair = defaultdict(set)
    frame_to_tracklets = defaultdict(list)
    for idx, tl in enumerate(tracklets):
        for f in tl["frames"]:
            frame_to_tracklets[f].append(idx)

    for f, idxs in frame_to_tracklets.items():
        for i in range(len(idxs)):
            for j in range(i + 1, len(idxs)):
                a, b = tracklets[idxs[i]], tracklets[idxs[j]]
                if a["class"] != b["class"]:
                    continue
                if iou(a["bbox_by_frame"][f], b["bbox_by_frame"][f]) >= iou_thresh:
                    pair = tuple(sorted((idxs[i], idxs[j])))
                    overlap_frames_by_pair[pair].add(f)

    split_points = defaultdict(set)
    for (i, j), frames in overlap_frames_by_pair.items():
        frames = sorted(frames)
        runs, current = [], [frames[0]]
        for f in frames[1:]:
            if f - current[-1] <= merge_gap:
                current.append(f)
            else:
                runs.append(current)
                current = [f]
        runs.append(current)
        for run in runs:
            split_points[i].add(run[0] - 1)
            split_points[i].add(run[-1])
            split_points[j].add(run[0] - 1)
            split_points[j].add(run[-1])
    return split_points


def slice_tracklet(tl, frames):
    return {
        "frames": frames,
        "bbox_by_frame": {f: tl["bbox_by_frame"][f] for f in frames},
        "conf_by_frame": {f: tl["conf_by_frame"][f] for f in frames},
        "class_by_frame": {f: tl["class_by_frame"][f] for f in frames},
        "embedding_by_frame": {f: tl["embedding_by_frame"][f] for f in frames},
        "class": tl["class"],
    }


def apply_splits(tracklets, split_points):
    new_tracklets = []
    for idx, tl in enumerate(tracklets):
        cuts = set(split_points.get(idx, []))
        if not cuts:
            new_tracklets.append(tl)
            continue
        chunks, current = [], []
        for f in tl["frames"]:
            current.append(f)
            if f in cuts:
                chunks.append(current)
                current = []
        if current:
            chunks.append(current)
        for chunk in chunks:
            if chunk:
                new_tracklets.append(slice_tracklet(tl, chunk))
    return new_tracklets


def fit_velocity(frames, bbox_by_frame):
    if len(frames) < 2:
        return None
    farr = np.array(frames, dtype=np.float64)
    centers = np.array([bbox_center(bbox_by_frame[f]) for f in frames])
    A = np.vstack([farr, np.ones_like(farr)]).T
    vx, cx = np.linalg.lstsq(A, centers[:, 0], rcond=None)[0]
    vy, cy = np.linalg.lstsq(A, centers[:, 1], rcond=None)[0]
    return np.array([vx, vy])


def compute_tracklet_features(tl, window):
    frames = tl["frames"]
    head_frames = frames[:window]
    tail_frames = frames[-window:]

    head_embs = [tl["embedding_by_frame"][f] for f in head_frames if tl["embedding_by_frame"].get(f) is not None]
    tail_embs = [tl["embedding_by_frame"][f] for f in tail_frames if tl["embedding_by_frame"].get(f) is not None]
    tl["head_emb"] = np.mean(head_embs, axis=0) if head_embs else None
    tl["tail_emb"] = np.mean(tail_embs, axis=0) if tail_embs else None

    tl["head_vel"] = fit_velocity(head_frames, tl["bbox_by_frame"])
    tl["tail_vel"] = fit_velocity(tail_frames, tl["bbox_by_frame"])
    tl["head_pos"] = bbox_center(tl["bbox_by_frame"][frames[0]])
    tl["tail_pos"] = bbox_center(tl["bbox_by_frame"][frames[-1]])


def link_score(ti, tj, motion_weight, motion_norm_px, max_gap):
    gap = tj["frames"][0] - ti["frames"][-1]
    if gap <= 0 or gap > max_gap:
        return None

    if ti["tail_emb"] is None or tj["head_emb"] is None:
        appearance_sim = 0.0
    else:
        appearance_sim = float(np.dot(ti["tail_emb"], tj["head_emb"]))

    if ti["tail_vel"] is not None:
        predicted = ti["tail_pos"] + ti["tail_vel"] * gap
    else:
        predicted = ti["tail_pos"]
    dist = float(np.linalg.norm(predicted - tj["head_pos"]))
    motion_penalty = min(dist / motion_norm_px, 1.0)

    return appearance_sim - motion_weight * motion_penalty


def link_tracklets_globally(tracklets, max_gap, min_link_score, motion_weight, motion_norm_px, debug=False):
    """One Hungarian-algorithm assignment problem PER CLASS, over ALL tracklets of that class
    at once -- not a sliding window. This is what makes it a genuinely global solution: a
    contact at frame 200 can be correctly resolved using appearance evidence from frame 5000 if
    that's where the clearest signal happens to be."""
    links = {}
    by_class = defaultdict(list)
    for idx, tl in enumerate(tracklets):
        by_class[tl["class"]].append(idx)

    for cls, idxs in by_class.items():
        idxs_by_end = sorted(idxs, key=lambda i: tracklets[i]["frames"][-1])
        idxs_by_start = sorted(idxs, key=lambda i: tracklets[i]["frames"][0])
        n, m = len(idxs_by_end), len(idxs_by_start)
        if n == 0 or m == 0:
            continue

        BIG = 1e6
        cost = np.full((n, m), BIG)
        for a, i in enumerate(idxs_by_end):
            for b, j in enumerate(idxs_by_start):
                if i == j:
                    continue
                score = link_score(tracklets[i], tracklets[j], motion_weight, motion_norm_px, max_gap)
                if score is not None and score >= min_link_score:
                    cost[a, b] = -score

        row_ind, col_ind = linear_sum_assignment(cost)
        for a, b in zip(row_ind, col_ind):
            if cost[a, b] >= BIG:
                continue
            i, j = idxs_by_end[a], idxs_by_start[b]
            links[i] = j
            if debug:
                print(f"  [link] class={cls}: tracklet {i} (ends frame {tracklets[i]['frames'][-1]}) "
                      f"-> tracklet {j} (starts frame {tracklets[j]['frames'][0]}), score={-cost[a,b]:.3f}")

    return links


class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx != ry:
            self.parent[rx] = ry


print("Core functions ready.")


Core functions ready.


## Cell 5 — Unit test: global linking reconnects a fragmented player and rejects a decoy
Synthetic scenario, independent of the video/detection cache: one real player is fragmented into
3 tracklets by two separate contact events; a second, unrelated player's single tracklet exists
in the same time range. Confirms the global linker chains the 3 real fragments into one identity
and does NOT accidentally link the decoy into that chain. Run this before spending GPU time on
the full video.

In [28]:
def _run_global_linking_unit_test():
    print("Running global linking unit test...")

    rng = np.random.default_rng(7)
    emb_player = rng.normal(size=512); emb_player /= np.linalg.norm(emb_player)
    emb_decoy  = rng.normal(size=512); emb_decoy  /= np.linalg.norm(emb_decoy)

    def make(frames, x_start, vx, emb, cls="player"):
        bbox_by_frame, conf_by_frame, class_by_frame, embedding_by_frame = {}, {}, {}, {}
        for k, f in enumerate(frames):
            x = x_start + vx * k
            bbox_by_frame[f] = [x, 100, x + 40, 180]
            conf_by_frame[f] = 0.9
            class_by_frame[f] = cls
            embedding_by_frame[f] = emb
        return {
            "frames": frames, "bbox_by_frame": bbox_by_frame, "conf_by_frame": conf_by_frame,
            "class_by_frame": class_by_frame, "embedding_by_frame": embedding_by_frame, "class": cls,
        }

    frag1 = make(list(range(0, 20)), x_start=100, vx=2, emb=emb_player)
    frag2 = make(list(range(40, 60)), x_start=100 + 2*19 + 2*20, vx=2, emb=emb_player)
    frag3 = make(list(range(90, 110)), x_start=100 + 2*19 + 2*20 + 2*19 + 2*30, vx=2, emb=emb_player)
    decoy = make(list(range(45, 65)), x_start=500, vx=-1, emb=emb_decoy)

    test_tracklets = [frag1, frag2, frag3, decoy]
    for tl in test_tracklets:
        compute_tracklet_features(tl, window=8)

    links = link_tracklets_globally(test_tracklets, max_gap=50, min_link_score=0.3,
                                     motion_weight=0.3, motion_norm_px=300.0, debug=False)

    test_uf = UnionFind(len(test_tracklets))
    for i, j in links.items():
        test_uf.union(i, j)
    groups = defaultdict(list)
    for idx in range(len(test_tracklets)):
        groups[test_uf.find(idx)].append(idx)

    print(f"  links: {links}")
    print(f"  groups: {dict(groups)}")

    group_sizes = sorted(len(v) for v in groups.values())
    assert group_sizes == [1, 3], f"FAIL: expected groups of size [1,3], got {group_sizes}"
    real_group = [v for v in groups.values() if len(v) == 3][0]
    assert set(real_group) == {0, 1, 2}, f"FAIL: wrong tracklets grouped: {real_group}"

    print("PASS: 3 fragments of the real player correctly chained; decoy correctly excluded.\n")


_run_global_linking_unit_test()


Running global linking unit test...
  links: {0: 1, 1: 2}
  groups: {2: [0, 1, 2], 3: [3]}
PASS: 3 fragments of the real player correctly chained; decoy correctly excluded.



## Cell 6 — Raw tracking pass (single causal pass over the real video)
No gallery, no swap corrector, no online ID resolution -- just record BoT-SORT's own raw track
ID, bbox, and embedding per frame. Everything from here on operates on this raw output using the
functions defined in Cell 4.

In [29]:
raw_tracks_by_frame = {}   # frame_idx -> [{"raw_track_id":, "bbox":, "conf":, "class":, "embedding":}]

cap = cv2.VideoCapture(VIDEO_PATH)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_dets = detection_cache.get(frame_idx, [])
    person_dets = [d for d in frame_dets if d["class"] in CLASS_TO_ID]

    dets_array = (np.array([[*d["bbox"], d["conf"], CLASS_TO_ID[d["class"]]] for d in person_dets], dtype=np.float64)
                  if person_dets else np.empty((0, 6), dtype=np.float64))

    tracked = tracker.update(dets_array, frame)

    embedding_lookup = {}
    for strack in tracker.active_tracks:
        if strack.curr_feat is not None:
            emb = strack.curr_feat
            norm = np.linalg.norm(emb)
            embedding_lookup[strack.id] = emb / norm if norm > 0 else emb

    frame_entries = []
    for x1, y1, x2, y2, raw_tid, conf, cls_id, det_ind in tracked:
        raw_tid = int(raw_tid)
        bbox = [float(x1), float(y1), float(x2), float(y2)]
        cls = person_dets[int(det_ind)]["class"]   # raw per-frame class, not boxmot's smoothed label
        embedding = embedding_lookup.get(raw_tid)
        frame_entries.append({"raw_track_id": raw_tid, "bbox": bbox, "conf": float(conf),
                               "class": cls, "embedding": embedding})

    raw_tracks_by_frame[frame_idx] = frame_entries

    if frame_idx % 200 == 0:
        print(f"frame {frame_idx}/{total_frames} | active raw tracks: {len(frame_entries)}")

    frame_idx += 1

cap.release()
print(f"\nDone. Processed {frame_idx} frames of raw tracking.")


frame 0/3001 | active raw tracks: 23
frame 200/3001 | active raw tracks: 24
frame 400/3001 | active raw tracks: 24
frame 600/3001 | active raw tracks: 24
frame 800/3001 | active raw tracks: 24
frame 1000/3001 | active raw tracks: 24
frame 1200/3001 | active raw tracks: 24
frame 1400/3001 | active raw tracks: 24
frame 1600/3001 | active raw tracks: 23
frame 1800/3001 | active raw tracks: 24
frame 2000/3001 | active raw tracks: 24
frame 2200/3001 | active raw tracks: 24
frame 2400/3001 | active raw tracks: 23
frame 2600/3001 | active raw tracks: 24
frame 2800/3001 | active raw tracks: 23


[opus @ 0x1dd177c0] Error parsing Opus packet header.


frame 3000/3001 | active raw tracks: 25

Done. Processed 3001 frames of raw tracking.


## Cell 7 — Build initial tracklets, split at contact points

In [30]:
initial_tracklets = build_initial_tracklets(raw_tracks_by_frame)
print(f"Initial tracklets (from raw track IDs): {len(initial_tracklets)}")

split_points = find_contact_split_points(initial_tracklets, CONTACT_IOU_THRESH)
split_tracklets = apply_splits(initial_tracklets, split_points)
print(f"Tracklets after contact-based splitting: {len(split_tracklets)} "
      f"(from {len(initial_tracklets)}, {sum(len(v) for v in split_points.values())} split points applied)")

pre_filter_count = len(split_tracklets)
split_tracklets = [tl for tl in split_tracklets if len(tl["frames"]) >= MIN_TRACKLET_LEN]
print(f"Dropped {pre_filter_count - len(split_tracklets)} tracklets shorter than "
      f"MIN_TRACKLET_LEN={MIN_TRACKLET_LEN} frames (too short for reliable linking)")
print(f"Tracklets going into global linking: {len(split_tracklets)}")


Initial tracklets (from raw track IDs): 119
Tracklets after contact-based splitting: 385 (from 119, 325 split points applied)
Dropped 251 tracklets shorter than MIN_TRACKLET_LEN=20 frames (too short for reliable linking)
Tracklets going into global linking: 134


## Cell 8 — Compute head/tail features, then run global linking

In [31]:
for tl in split_tracklets:
    compute_tracklet_features(tl, EMBED_WINDOW)
print(f"Computed head/tail features for {len(split_tracklets)} tracklets.")

accepted_links = link_tracklets_globally(
    split_tracklets, MAX_LINK_GAP, MIN_LINK_SCORE, MOTION_WEIGHT, MOTION_NORM_PX, debug=DEBUG_LINKING
)
print(f"\nTotal accepted links: {len(accepted_links)}")


Computed head/tail features for 134 tracklets.
  [link] class=referee: tracklet 0 (ends frame 896) -> tracklet 1 (starts frame 898), score=0.902
  [link] class=referee: tracklet 1 (ends frame 1845) -> tracklet 129 (starts frame 2158), score=0.505
  [link] class=referee: tracklet 129 (ends frame 2626) -> tracklet 130 (starts frame 2648), score=0.907
  [link] class=player: tracklet 62 (ends frame 64) -> tracklet 63 (starts frame 72), score=0.781
  [link] class=player: tracklet 92 (ends frame 64) -> tracklet 93 (starts frame 71), score=0.792
  [link] class=player: tracklet 14 (ends frame 71) -> tracklet 15 (starts frame 76), score=0.847
  [link] class=player: tracklet 4 (ends frame 102) -> tracklet 5 (starts frame 105), score=0.738
  [link] class=player: tracklet 3 (ends frame 104) -> tracklet 110 (starts frame 111), score=0.793
  [link] class=player: tracklet 75 (ends frame 120) -> tracklet 76 (starts frame 124), score=0.776
  [link] class=player: tracklet 81 (ends frame 120) -> tracklet

## Cell 9 — Union accepted links into final global identities

In [32]:
uf = UnionFind(len(split_tracklets))
for i, j in accepted_links.items():
    uf.union(i, j)

chain_members = defaultdict(list)
for idx in range(len(split_tracklets)):
    chain_members[uf.find(idx)].append(idx)

canonical_id_of_tracklet = {}
canonical_class = {}
next_id = 1
for root, members in chain_members.items():
    for m in members:
        canonical_id_of_tracklet[m] = next_id
    canonical_class[next_id] = split_tracklets[members[0]]["class"]
    next_id += 1

chain_lengths = [len(members) for members in chain_members.values()]
print(f"Final global identities: {len(chain_members)}")
print(f"Chain length distribution: 1 tracklet={sum(1 for c in chain_lengths if c==1)}, "
      f"2={sum(1 for c in chain_lengths if c==2)}, 3+={sum(1 for c in chain_lengths if c>=3)}")

by_class_counts = defaultdict(int)
for cid, cls in canonical_class.items():
    by_class_counts[cls] += 1
print("Identity counts by class:", dict(by_class_counts))
for cls, expected in MAX_IDS_PER_CLASS_EXPECTED.items():
    actual = by_class_counts.get(cls, 0)
    if actual > expected:
        print(f"  [check] {cls}: {actual} identities, expected <= {expected} -- consider raising "
              f"MAX_LINK_GAP or lowering MIN_LINK_SCORE, or check [link] output for rejected pairs")


Final global identities: 27
Chain length distribution: 1 tracklet=4, 2=1, 3+=22
Identity counts by class: {'referee': 2, 'player': 22, 'goalkeeper': 3}
  [check] goalkeeper: 3 identities, expected <= 2 -- consider raising MAX_LINK_GAP or lowering MIN_LINK_SCORE, or check [link] output for rejected pairs


## Cell 10 — Rebuild the final per-frame tracking cache with global IDs

In [33]:
final_tracking_cache = defaultdict(lambda: {"ball": None, "tracks": []})

for idx, tl in enumerate(split_tracklets):
    cid = canonical_id_of_tracklet[idx]
    for f in tl["frames"]:
        final_tracking_cache[f]["tracks"].append({
            "track_id": cid,
            "bbox": tl["bbox_by_frame"][f],
            "conf": tl["conf_by_frame"][f],
            "class": tl["class_by_frame"][f],
        })

for f in range(frame_idx):
    ball_det = next((d for d in detection_cache.get(f, []) if d["class"] == "ball"), None)
    final_tracking_cache.setdefault(f, {"ball": ball_det, "tracks": []})
    final_tracking_cache[f]["ball"] = ball_det

final_tracking_cache = dict(final_tracking_cache)
print(f"Rebuilt tracking cache: {len(final_tracking_cache)} frames.")


Rebuilt tracking cache: 3001 frames.


## Cell 11 — Ghost-track removal + ratio-aware class locking

In [34]:
id_frame_counts = defaultdict(int)
for data in final_tracking_cache.values():
    for t in data["tracks"]:
        id_frame_counts[t["track_id"]] += 1

ghost_ids = {tid for tid, count in id_frame_counts.items() if count < MIN_TRACK_LENGTH}
print(f"Ghost tracks dropped (< {MIN_TRACK_LENGTH} frames): {sorted(ghost_ids)}")

for data in final_tracking_cache.values():
    data["tracks"] = [t for t in data["tracks"] if t["track_id"] not in ghost_ids]

class_votes = defaultdict(lambda: defaultdict(int))
for data in final_tracking_cache.values():
    for t in data["tracks"]:
        class_votes[t["track_id"]][t["class"]] += 1

locked_class_by_id = {}
for tid, votes in class_votes.items():
    total = sum(votes.values())
    ref_votes = votes.get("referee", 0)
    gk_votes = votes.get("goalkeeper", 0)
    ref_confirmed = ref_votes >= MIN_CONFIRM_FRAMES_ABS and (ref_votes / total) >= MIN_CONFIRM_RATIO
    gk_confirmed = gk_votes >= MIN_CONFIRM_FRAMES_ABS and (gk_votes / total) >= MIN_CONFIRM_RATIO
    if ref_confirmed:
        locked_class_by_id[tid] = "referee"
    elif gk_confirmed:
        locked_class_by_id[tid] = "goalkeeper"
    else:
        locked_class_by_id[tid] = max(votes, key=votes.get)

print(f"\nFinal locked classes after ghost removal:\n")
for tid in sorted(locked_class_by_id):
    print(f"  id={tid:2d} | locked={locked_class_by_id[tid]} | votes={dict(class_votes[tid])} "
          f"| total_frames={id_frame_counts[tid]}")

final_counts = defaultdict(int)
for cls in locked_class_by_id.values():
    final_counts[cls] += 1
print(f"\nFinal identity counts: {dict(final_counts)}")


Ghost tracks dropped (< 300 frames): [25, 27]

Final locked classes after ghost removal:

  id= 1 | locked=referee | votes={'referee': 1815, 'player': 852} | total_frames=2667
  id= 2 | locked=referee | votes={'referee': 2317, 'player': 684} | total_frames=3001
  id= 3 | locked=player | votes={'player': 2979} | total_frames=2979
  id= 4 | locked=player | votes={'player': 2950, 'referee': 1} | total_frames=2951
  id= 5 | locked=player | votes={'player': 2959} | total_frames=2959
  id= 6 | locked=player | votes={'player': 2973} | total_frames=2973
  id= 7 | locked=referee | votes={'player': 2459, 'referee': 475} | total_frames=2934
  id= 8 | locked=player | votes={'player': 3000} | total_frames=3000
  id= 9 | locked=player | votes={'player': 2927} | total_frames=2927
  id=10 | locked=player | votes={'player': 3001} | total_frames=3001
  id=11 | locked=player | votes={'player': 2995} | total_frames=2995
  id=12 | locked=player | votes={'player': 2951} | total_frames=2951
  id=13 | locked=

## Cell 12 — Save final cache

In [35]:
with open(OUTPUT_TRACKING_CACHE_PATH, "wb") as f:
    pickle.dump({"tracking_cache": final_tracking_cache, "locked_class_by_id": locked_class_by_id}, f)

print(f"Saved to {OUTPUT_TRACKING_CACHE_PATH}")
print(f"Raw per-frame classes preserved in tracking_cache; final labels in locked_class_by_id.")

from IPython.display import FileLink
print(f"File size: {os.path.getsize(OUTPUT_TRACKING_CACHE_PATH) / 1024:.1f} KB")
FileLink(str(OUTPUT_TRACKING_CACHE_PATH))


Saved to barca_atletico_tracking_cache_final.pkl
Raw per-frame classes preserved in tracking_cache; final labels in locked_class_by_id.
File size: 4729.5 KB


/kaggle/working/barca_atletico_tracking_cache_final.pkl

## Cell 13 — Annotate full video from the finalized cache (frame numbers burned in)

In [36]:
def id_to_color(track_id):
    hue = (track_id * 0.618033988749895) % 1.0
    r, g, b = colorsys.hsv_to_rgb(hue, 0.65, 0.95)
    return (int(b * 255), int(g * 255), int(r * 255))

CLASS_COLOR_OVERRIDE = {"goalkeeper": (0, 200, 255), "referee": (0, 0, 255)}

def draw_pill_label(frame, text, x, y, color, font_scale=0.5, thickness=1):
    font = cv2.FONT_HERSHEY_SIMPLEX
    (tw, th), _ = cv2.getTextSize(text, font, font_scale, thickness)
    pad_x, pad_y = 6, 4
    x1, y1 = int(x), int(y - th - 2 * pad_y)
    x2, y2 = int(x + tw + 2 * pad_x), int(y)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, -1, lineType=cv2.LINE_AA)
    cv2.putText(frame, text, (x1 + pad_x, y2 - pad_y), font, font_scale, (255, 255, 255), thickness, cv2.LINE_AA)

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (frame_w, frame_h))

f_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    data = final_tracking_cache.get(f_idx, {"ball": None, "tracks": []})

    cv2.putText(frame, f"frame={f_idx}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                (0, 255, 255), 2, cv2.LINE_AA)

    for t in data["tracks"]:
        x1, y1, x2, y2 = [int(v) for v in t["bbox"]]
        tid = t["track_id"]
        cls = locked_class_by_id.get(tid, t["class"])
        color = CLASS_COLOR_OVERRIDE.get(cls, id_to_color(tid))

        if cls == "goalkeeper":
            cx, cy = (x1 + x2) // 2, y2
            cv2.ellipse(frame, (cx, cy), (int((x2 - x1) * 0.5), 8), 0, 0, 360, color, 3, cv2.LINE_AA)
        else:
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2, lineType=cv2.LINE_AA)
        draw_pill_label(frame, f"#{tid} {cls[:3].upper()}", x1, y1, color)

    if data["ball"] is not None:
        bx1, by1, bx2, by2 = [int(v) for v in data["ball"]["bbox"]]
        cx, cy = (bx1 + bx2) // 2, (by1 + by2) // 2
        cv2.circle(frame, (cx, cy), 6, (0, 255, 255), -1, lineType=cv2.LINE_AA)

    writer.write(frame)
    f_idx += 1

cap.release()
writer.release()
print(f"Annotated video saved to {OUTPUT_VIDEO_PATH} ({f_idx} frames).")


Annotated video saved to barca_atletico_annotated_final.mp4 (3001 frames).


[opus @ 0x1d9d21c0] Error parsing Opus packet header.
